In [5]:
import hashlib
import sys
import urllib.request
import time
from pyspark.sql import SparkSession
from pyspark.sql.functions import (
    col, regexp_extract, to_timestamp, concat_ws, nullif, lit
)
import glob

# ─────────────────────────────────────────────
# CONFIGURAÇÕES
# ─────────────────────────────────────────────
LOG_FILES = glob.glob("/opt/spark-data/*.log")

CH_HOST     = "clickhouse"
CH_PORT     = "8123"
CH_USER     = "admin"
CH_PASSWORD = "admin"
CH_DATABASE = "logs"

# ─────────────────────────────────────────────
# TIMER
# ─────────────────────────────────────────────
start = time.time()
print(f"[{start:.3f}][SPARK-JOB] Iniciando execução do Job...")
print(f"[SPARK-JOB] {len(LOG_FILES)} arquivo(s) encontrado(s).")

if not LOG_FILES:
    print("[SPARK-JOB] Nenhum arquivo de log encontrado. Encerrando.")
    sys.exit(0)

# ─────────────────────────────────────────────
# HELPER: requisição HTTP ao ClickHouse
# ─────────────────────────────────────────────
def ch_query(sql: str) -> str:
    url = f"http://{CH_HOST}:{CH_PORT}/?user={CH_USER}&password={CH_PASSWORD}"
    req = urllib.request.Request(url, sql.encode("utf-8"), method="POST")
    with urllib.request.urlopen(req, timeout=10) as resp:
        return resp.read().decode("utf-8").strip()

# ─────────────────────────────────────────────
# HELPER: calcular MD5 de um arquivo
# ─────────────────────────────────────────────
def calc_md5(path: str) -> str:
    md5 = hashlib.md5()
    with open(path, "rb") as f:
        for chunk in iter(lambda: f.read(8192), b""):
            md5.update(chunk)
    return md5.hexdigest()

# ─────────────────────────────────────────────
# STEP 1 — Garantir que as tabelas existem
# ─────────────────────────────────────────────
print("[SPARK-JOB] Verificando/criando tabelas no ClickHouse...")

ch_query(f"CREATE DATABASE IF NOT EXISTS {CH_DATABASE}")

ch_query(f"""
CREATE TABLE IF NOT EXISTS {CH_DATABASE}.router_logs
(
    timestamp DateTime,
    service   LowCardinality(String),
    level     LowCardinality(String),
    message   String,
    ip        Nullable(String),
    mac       Nullable(String)
)
ENGINE = MergeTree()
PARTITION BY toYYYYMM(timestamp)
ORDER BY (timestamp, service)
""")

ch_query(f"""
CREATE TABLE IF NOT EXISTS {CH_DATABASE}.file_hashes
(
    file_hash    String,
    file_name    String,
    processed_at DateTime DEFAULT now()
)
ENGINE = MergeTree()
ORDER BY file_hash
""")

print("[SPARK-JOB] Tabelas OK.")

# ─────────────────────────────────────────────
# STEP 2 — Filtrar apenas arquivos não processados
# ─────────────────────────────────────────────
pending = []
skipped = []
seen_hashes = {}

for path in sorted(LOG_FILES):
    file_hash = calc_md5(path)
    print(f"[SPARK-JOB] {path} → MD5: {file_hash}")

    try:
        result = ch_query(
            f"SELECT count() FROM {CH_DATABASE}.file_hashes "
            f"WHERE file_hash = '{file_hash}'"
        )
        if int(result) > 0:
            skipped.append(path)
            print(f"  ↳ Já processado (hash no banco). Pulando.")
        elif file_hash in seen_hashes:
            skipped.append(path)
            print(f"  ↳ Hash duplicado de '{seen_hashes[file_hash]}'. Pulando.")
        else:
            seen_hashes[file_hash] = path
            pending.append((path, file_hash))
            print(f"  ↳ Novo arquivo. Na fila.")
    except Exception as e:
        print(f"[SPARK-JOB][ERRO] Falha ao consultar ClickHouse para {path}: {e}")
        sys.exit(1)

if not pending:
    print("[SPARK-JOB] Nenhum arquivo novo para processar. Encerrando.")
    end = time.time()
    print(f"  Tempo total: {end - start:.2f}s")
    sys.exit(0)

# ─────────────────────────────────────────────
# STEP 3 — Iniciar Spark
# ─────────────────────────────────────────────
packages = ",".join([
    "com.clickhouse.spark:clickhouse-spark-runtime-3.5_2.12:0.10.0",
    "com.clickhouse:clickhouse-client:0.7.0",
    "com.clickhouse:clickhouse-http-client:0.7.0",
    "org.apache.httpcomponents.client5:httpclient5:5.2.1",
])

spark = (
    SparkSession.builder
    .config("spark.jars.packages", packages)
    .config("spark.sql.shuffle.partitions", "8")
    .config("spark.default.parallelism", "8")
    .config("spark.sql.adaptive.enabled", "true")
    .config("spark.sql.adaptive.coalescePartitions.enabled", "true")
    .config("spark.sql.adaptive.skewJoin.enabled", "true")
    .config("spark.sql.autoBroadcastJoinThreshold", "10mb")
    .config("spark.shuffle.compress", "true")
    .config("spark.io.compression.codec", "lz4")
    .config("spark.serializer", "org.apache.spark.serializer.KryoSerializer")
    .config("spark.kryoserializer.buffer.max", "512m")
    .config("spark.clickhouse.ignoreUnsupportedTransform", "true")
    .getOrCreate()
)

ch_conf = {
    "spark.sql.catalog.clickhouse":           "com.clickhouse.spark.ClickHouseCatalog",
    "spark.sql.catalog.clickhouse.host":      CH_HOST,
    "spark.sql.catalog.clickhouse.protocol":  "http",
    "spark.sql.catalog.clickhouse.http_port": CH_PORT,
    "spark.sql.catalog.clickhouse.user":      CH_USER,
    "spark.sql.catalog.clickhouse.password":  CH_PASSWORD,
    "spark.sql.catalog.clickhouse.database":  CH_DATABASE,
    "spark.clickhouse.write.format":          "json",
    "spark.clickhouse.write.batchSize":       "100000",
    "spark.clickhouse.write.maxRetries":      "3",
}
for k, v in ch_conf.items():
    spark.conf.set(k, v)

# ─────────────────────────────────────────────
# REGEX PATTERNS
# ─────────────────────────────────────────────
LOG_PATTERN = r"^(\w{3}/\d{2}/\d{4}) (\d{2}:\d{2}:\d{2}) ([\w\-]+),(\w+): (.*)$"
IP_PATTERN  = r"(?:^|\s)ip\s+(\d{1,3}(?:\.\d{1,3}){3})(?:\s|$)"
MAC_PATTERN = r"(?:^|\s)mac\s+([0-9A-Fa-f]{2}(?::[0-9A-Fa-f]{2}){5})(?:\s|$)"

# ─────────────────────────────────────────────
# STEP 4 — Processar cada arquivo pendente
# ─────────────────────────────────────────────
failed = []

for path, file_hash in pending:
    print(f"\n[SPARK-JOB] Processando: {path}")
    try:
        df = spark.read.text(path)

        parsed = df.select(
            regexp_extract("value", LOG_PATTERN, 1).alias("date"),
            regexp_extract("value", LOG_PATTERN, 2).alias("time"),
            regexp_extract("value", LOG_PATTERN, 3).alias("service"),
            regexp_extract("value", LOG_PATTERN, 4).alias("level"),
            regexp_extract("value", LOG_PATTERN, 5).alias("message"),
        )

        structured = (
            parsed
            .withColumn(
                "timestamp",
                to_timestamp(concat_ws(" ", col("date"), col("time")), "MMM/dd/yyyy HH:mm:ss")
            )
            .withColumn("ip",  nullif(regexp_extract(col("message"), IP_PATTERN,  1), lit("")))
            .withColumn("mac", nullif(regexp_extract(col("message"), MAC_PATTERN, 1), lit("")))
            .select("timestamp", "service", "level", "message", "ip", "mac")
        )

        structured.writeTo("clickhouse.logs.router_logs").append()

        ch_query(
            f"INSERT INTO {CH_DATABASE}.file_hashes (file_hash, file_name, processed_at) "
            f"VALUES ('{file_hash}', '{path}', now())"
        )
        print(f"  ↳ Concluído e hash registrado.")

    except Exception as e:
        print(f"  ↳ [ERRO] Falha ao processar {path}: {e}")
        failed.append(path)

# ─────────────────────────────────────────────
# FIM
# ─────────────────────────────────────────────
spark.stop()
end = time.time()

print(f"\n[SPARK-JOB] ═══════════ Relatório Final ═══════════")
print(f"  Total encontrados:  {len(LOG_FILES)}")
print(f"  Já processados:     {len(skipped)}")
print(f"  Processados agora:  {len(pending) - len(failed)}")
print(f"  Com erro:           {len(failed)}")
if failed:
    for f in failed:
        print(f"    ✗ {f}")
print(f"  Tempo total:        {end - start:.2f}s")
print(f"[{end:.3f}][SPARK-JOB] Job finalizado.")

if failed:
    sys.exit(1)

[1777298270.956][SPARK-JOB] Iniciando execução do Job...
[SPARK-JOB] 2 arquivo(s) encontrado(s).
[SPARK-JOB] /opt/spark-data/logs copy.log → MD5: 183396a394e0502b2e19522328426044
  ↳ Novo arquivo. Na fila.
[SPARK-JOB] /opt/spark-data/logs.log → MD5: 3db4413b19c4565f279138bbbdc7c880
  ↳ Novo arquivo. Na fila.

[SPARK-JOB] Processando: /opt/spark-data/logs copy.log


26/04/27 13:57:58 WARN ExprUtils: Ignoring unsupported ClickHouse partition/sharding expression: FuncExpr[toYYYYMM(timestamp)]. Spark-side repartitioning will be skipped for this expression. To fail on unsupported expressions, set spark.clickhouse.ignoreUnsupportedTransform=false. Reason:  [-1] Unsupported ClickHouse expression: FuncExpr[toYYYYMM(timestamp)]
26/04/27 13:57:58 WARN ExprUtils: Ignoring unsupported ClickHouse partition/sharding expression: FuncExpr[toYYYYMM(timestamp)]. Spark-side repartitioning will be skipped for this expression. To fail on unsupported expressions, set spark.clickhouse.ignoreUnsupportedTransform=false. Reason:  [-1] Unsupported ClickHouse expression: FuncExpr[toYYYYMM(timestamp)]
                                                                                

  ↳ Concluído e hash registrado.

[SPARK-JOB] Processando: /opt/spark-data/logs.log


26/04/27 13:58:01 WARN ExprUtils: Ignoring unsupported ClickHouse partition/sharding expression: FuncExpr[toYYYYMM(timestamp)]. Spark-side repartitioning will be skipped for this expression. To fail on unsupported expressions, set spark.clickhouse.ignoreUnsupportedTransform=false. Reason:  [-1] Unsupported ClickHouse expression: FuncExpr[toYYYYMM(timestamp)]
26/04/27 13:58:01 WARN ExprUtils: Ignoring unsupported ClickHouse partition/sharding expression: FuncExpr[toYYYYMM(timestamp)]. Spark-side repartitioning will be skipped for this expression. To fail on unsupported expressions, set spark.clickhouse.ignoreUnsupportedTransform=false. Reason:  [-1] Unsupported ClickHouse expression: FuncExpr[toYYYYMM(timestamp)]
                                                                                

  ↳ Concluído e hash registrado.

[SPARK-JOB] ═══════════ Relatório Final ═══════════
  Total encontrados:  2
  Já processados:     0
  Processados agora:  2
  Com erro:           0
  Tempo total:        13.96s
[1777298284.919][SPARK-JOB] Job finalizado.
